### **IMPORTO LIBRERIE**

In [ ]:
from pathlib import Path

import pandas as pd

### **IMPORTO DATI**

In [ ]:
#Importo i dataframe precedentemente puliti per l'analisi
df_software = pd.read_csv("datas/processed/dataset_software.csv")
df_hardware = pd.read_csv("datas/processed/dataset_hardware.csv")
df_software_test = pd.read_csv("datas/processed/dataset_software_test.csv")
df_hardware_test = pd.read_csv("datas/processed/dataset_hardware_test.csv")

### **STATISTICA DESCRITTIVA**

In [ ]:
colonne_attivo = [
    "Totale Attivo migl USD 2024",
    "Totale Attivo migl USD 2023",
    "Totale Attivo migl USD 2022",
    "Totale Attivo migl USD 2021",
    "Totale Attivo migl USD 2020",
]

colonne_descrittive = [
    "roe_medio",
    "roa_medio",
    "debt_equity_medio",
    "liquidita_media",
    "attivo_medio",
]

etichette_variabili = {
    "roe_medio": "ROE medio",
    "roa_medio": "ROA medio",
    "debt_equity_medio": "Debt/Equity medio",
    "liquidita_media": "Indice di liquidità medio",
    "attivo_medio": "Totale attivo medio (migliaia di USD)",
}

etichette_statistiche = {
    "mean": "Media",
    "median": "Mediana",
    "std": "Deviazione standard",
    "min": "Minimo",
    "max": "Massimo",
}

cartella_export_descrittiva = Path("statistica_descrittiva_export/tex")


def aggiungi_attivo_medio(df):
    df = df.copy()
    df["attivo_medio"] = df[colonne_attivo].astype(float).mean(axis=1, skipna=True)
    return df


def crea_tabella_statistica_descrittiva(df, titolo, nome_file):
    tabella = df[colonne_descrittive].agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ]).T

    tabella = tabella.rename(
        index=etichette_variabili,
        columns=etichette_statistiche
    )
    tabella.index.name = "Indicatore"

    cartella_export_descrittiva.mkdir(exist_ok=True)
    percorso_tex = cartella_export_descrittiva / nome_file
    tabella_export = tabella.reset_index()
    tabella_export.to_latex(
        percorso_tex,
        caption=titolo,
        label=f"tab:{percorso_tex.stem}",
        escape=True,
        index=False,
        float_format="{:,.2f}".format,
        column_format="lrrrrr",
    )

    return tabella


def formatta_tabella_descrittiva(tabella, titolo):
    return (
        tabella.style
        .set_caption(titolo)
        .format("{:,.2f}")
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"), ("font-size", "1.1em")]},
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "right")]},
        ])
    )


df_software = aggiungi_attivo_medio(df_software)
df_hardware = aggiungi_attivo_medio(df_hardware)
df_completo = pd.concat([df_software, df_hardware], ignore_index=True)

statistica_descrittiva_totale = crea_tabella_statistica_descrittiva(
    df_completo,
    "Statistiche descrittive - Campione totale",
    "statistica_descrittiva_totale.tex"
)

statistica_descrittiva_hardware = crea_tabella_statistica_descrittiva(
    df_hardware,
    "Statistiche descrittive - Hardware",
    "statistica_descrittiva_hardware.tex"
)

statistica_descrittiva_software = crea_tabella_statistica_descrittiva(
    df_software,
    "Statistiche descrittive - Software",
    "statistica_descrittiva_software.tex"
)

stile_statistica_descrittiva_totale = formatta_tabella_descrittiva(
    statistica_descrittiva_totale,
    "Statistiche descrittive - Campione totale"
)
stile_statistica_descrittiva_hardware = formatta_tabella_descrittiva(
    statistica_descrittiva_hardware,
    "Statistiche descrittive - Hardware"
)
stile_statistica_descrittiva_software = formatta_tabella_descrittiva(
    statistica_descrittiva_software,
    "Statistiche descrittive - Software"
)

percorsi_export_descrittiva = sorted(cartella_export_descrittiva.glob("*.tex"))
percorsi_export_descrittiva

In [ ]:
stile_statistica_descrittiva_totale

In [ ]:
stile_statistica_descrittiva_hardware

In [ ]:
stile_statistica_descrittiva_software

Dato che il ROE medio e mediano è negativo, penso sia interessante analizzare il modo in cui è distribuito (prevedendo ovviamente un'asimmetria a sinistra).

In [ ]:
def mostra_istogramma_roe(df, nome_database, colonna_roe="roe_medio", bins=30):
    import matplotlib.pyplot as plt

    roe = df[colonna_roe].dropna()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(roe, bins=bins, color="#4C78A8", edgecolor="black", alpha=0.85)
    ax.axvline(roe.mean(), color="#E45756", linestyle="--", linewidth=2, label=f"Media: {roe.mean():.2f}")
    ax.axvline(roe.median(), color="#54A24B", linestyle="-", linewidth=2, label=f"Mediana: {roe.median():.2f}")
    ax.set_title(f"Distribuzione del ROE - {nome_database}")
    ax.set_xlabel("ROE medio")
    ax.set_ylabel("Numero di aziende")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.show()

    return fig, ax


mostra_istogramma_roe(
    df_software,
    "Software"
);

mostra_istogramma_roe(
    df_hardware,
    "Hardware"
);